In [ ]:
# =================================================================
# 1. CONFIGURATION & IMPORTS
# =================================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import step1_loading
import step2_preprocessing
import step3_epoching
import step4_pseudotrials
import step5_decoding
import step6_visualization
import step7_comparison

# Configuration
SUBJECT_ID = "sub-01"  # Change this to test different subjects
BASIC_PATH = 'project/ds006761'

print(f"--- TESTING SINGLE SUBJECT: {SUBJECT_ID} ---")

In [ ]:
# =================================================================
# 2. LOAD DATA & DETERMINE WINNER
# =================================================================
print(f"\nLoading data for {SUBJECT_ID}...")

# Load raw data
raw_p1_full, raw_p2_full = step1_loading.load_and_split_data(SUBJECT_ID, BASIC_PATH)

# Determine game winner
try:
    ev_file = f'{BASIC_PATH}/{SUBJECT_ID}/eeg/{SUBJECT_ID}_task-RPS_events.tsv'
    ev_df = pd.read_csv(ev_file, sep='\t')
    
    # Calculate scores
    p1_vals = ev_df['player1_resp'].values
    p2_vals = ev_df['player2_resp'].values
    
    # Count wins excluding NaNs
    valid_mask = np.isfinite(p1_vals) & np.isfinite(p2_vals)
    diff = (p1_vals[valid_mask] - p2_vals[valid_mask]) % 3
    
    p1_wins = np.sum(diff == 1)
    p2_wins = np.sum(diff == 2)
    
    subject_winner = 1 if p1_wins >= p2_wins else 2
    print(f"Game Outcome: P1({p1_wins}) vs P2({p2_wins}) => Winner is Player {subject_winner}")
    
except Exception as e:
    print(f"Warning: Could not determine winner, defaulting to P1: {e}")
    subject_winner = 1

In [ ]:
# =================================================================
# 3. PROCESS PLAYER 1 (STEP-BY-STEP)
# =================================================================
print(f"\n{'='*60}")
print("PROCESSING PLAYER 1")
print('='*60)

player_num = 1
raw_data = raw_p1_full

# A. PREPROCESSING
print("\nA. Preprocessing...")
raw_clean = step2_preprocessing.run_preprocessing(raw_data, SUBJECT_ID, BASIC_PATH, player_num)

# B. EPOCHING
print("\nB. Epoching...")
epochs_tuple, full_df = step3_epoching.run_epoching(raw_clean, SUBJECT_ID, BASIC_PATH, player_num)
print(f"   Number of trials after block removal: {len(full_df)}")

# C. TIME BINNING
print("\nC. Time Binning (creating 20-bin time course)...")
epochs_binned = step4_pseudotrials.create_pseudo_trials(epochs_tuple)
print(f"   Binned shape: {epochs_binned.shape}")

# D. GET LABELS
labels = full_df[f'player{player_num}_resp'].values
print(f"\nD. Labels: Rock={sum(labels==1)}, Paper={sum(labels==2)}, Scissors={sum(labels==3)}")

# E. CREATE PSEUDO-TRIALS (Trial Averaging)
print("\nE. Creating pseudo-trials (averaging 4 trials, 20 repeats)...")
X_pseudo, y_pseudo = step4_pseudotrials.create_pseudotrials_paper_method(
    epochs_binned, labels, 
    n_average=4, 
    n_repeats=20,
    random_seed=1  # Paper uses 1
)
print(f"   Pseudo-trials shape: {X_pseudo.shape}")
print(f"   Pseudo-labels: Rock={sum(y_pseudo==1)}, Paper={sum(y_pseudo==2)}, Scissors={sum(y_pseudo==3)}")

# Store Player 1 results for later comparison
player1_results = {
    'epochs_binned': epochs_binned,
    'full_df': full_df,
    'X_pseudo': X_pseudo,
    'y_pseudo': y_pseudo,
    'is_winner': (player_num == subject_winner)
}

In [ ]:
# =================================================================
# 4. DECODE PLAYER 1 - OWN CURRENT RESPONSE
# =================================================================
print(f"\n{'='*60}")
print("DECODING PLAYER 1 - OWN CURRENT RESPONSE")
print('='*60)

# Run SVM decoding
times, scores, std_scores, fold_accuracies = step5_decoding.run_svm_decoding_with_pseudotrials(
    player1_results['X_pseudo'], 
    player1_results['y_pseudo'],
    n_folds=10,
    n_average=4,
    n_repeats=20
)

if scores is not None:
    # Plot results
    step6_visualization.plot_paper_replication(times, scores, f"Player {player_num} - Own Current Response")
    
    # Store scores
    player1_scores = scores
    print(f"\n✅ Player 1 decoding complete!")
    print(f"   Mean accuracy: {np.mean(scores):.2f}%")
    print(f"   Peak accuracy: {np.max(scores):.2f}% at bin {np.argmax(scores)}")
else:
    print("❌ Player 1 decoding failed")

In [ ]:
# =================================================================
# 5. PROCESS PLAYER 2 (STEP-BY-STEP)
# =================================================================
print(f"\n{'='*60}")
print("PROCESSING PLAYER 2")
print('='*60)

player_num = 2
raw_data = raw_p2_full

# A. PREPROCESSING
print("\nA. Preprocessing...")
raw_clean = step2_preprocessing.run_preprocessing(raw_data, SUBJECT_ID, BASIC_PATH, player_num)

# B. EPOCHING
print("\nB. Epoching...")
epochs_tuple, full_df = step3_epoching.run_epoching(raw_clean, SUBJECT_ID, BASIC_PATH, player_num)
print(f"   Number of trials after block removal: {len(full_df)}")

# C. TIME BINNING
print("\nC. Time Binning (creating 20-bin time course)...")
epochs_binned = step4_pseudotrials.create_pseudo_trials(epochs_tuple)
print(f"   Binned shape: {epochs_binned.shape}")

# D. GET LABELS
labels = full_df[f'player{player_num}_resp'].values
print(f"\nD. Labels: Rock={sum(labels==1)}, Paper={sum(labels==2)}, Scissors={sum(labels==3)}")

# E. CREATE PSEUDO-TRIALS (Trial Averaging)
print("\nE. Creating pseudo-trials (averaging 4 trials, 20 repeats)...")
X_pseudo, y_pseudo = step4_pseudotrials.create_pseudotrials_paper_method(
    epochs_binned, labels, 
    n_average=4, 
    n_repeats=20,
    random_seed=1  # Paper uses 1
)
print(f"   Pseudo-trials shape: {X_pseudo.shape}")
print(f"   Pseudo-labels: Rock={sum(y_pseudo==1)}, Paper={sum(y_pseudo==2)}, Scissors={sum(y_pseudo==3)}")

# Store Player 2 results for later comparison
player2_results = {
    'epochs_binned': epochs_binned,
    'full_df': full_df,
    'X_pseudo': X_pseudo,
    'y_pseudo': y_pseudo,
    'is_winner': (player_num == subject_winner)
}

In [ ]:
# =================================================================
# 6. DECODE PLAYER 2 - OWN CURRENT RESPONSE
# =================================================================
print(f"\n{'='*60}")
print("DECODING PLAYER 2 - OWN CURRENT RESPONSE")
print('='*60)

# Run SVM decoding
times, scores, std_scores, fold_accuracies = step5_decoding.run_svm_decoding_with_pseudotrials(
    player2_results['X_pseudo'], 
    player2_results['y_pseudo'],
    n_folds=10,
    n_average=4,
    n_repeats=20
)

if scores is not None:
    # Plot results
    step6_visualization.plot_paper_replication(times, scores, f"Player {player_num} - Own Current Response")
    
    # Store scores
    player2_scores = scores
    print(f"\n✅ Player 2 decoding complete!")
    print(f"   Mean accuracy: {np.mean(scores):.2f}%")
    print(f"   Peak accuracy: {np.max(scores):.2f}% at bin {np.argmax(scores)}")
else:
    print("❌ Player 2 decoding failed")

In [ ]:
# =================================================================
# 7. COMPARATIVE ANALYSIS FOR BOTH PLAYERS
# =================================================================
print(f"\n{'='*60}")
print("COMPARATIVE ANALYSIS - 4 DECODING TASKS")
print('='*60)

# Initialize storage
group_comp_results = {
    'Winner': {'Own Current': [], 'Opponent Current': [], 'Own Previous': [], 'Opponent Previous': []},
    'Loser': {'Own Current': [], 'Opponent Current': [], 'Own Previous': [], 'Opponent Previous': []}
}

# Analyze Player 1
print(f"\nAnalyzing Player 1...")
player_num = 1
comp_results_p1 = step7_comparison.run_comparative_analysis(
    player1_results['epochs_binned'], 
    player1_results['full_df'], 
    player_num
)

# Determine group
group_key = 'Winner' if player1_results['is_winner'] else 'Loser'
for task_name, task_scores in comp_results_p1.items():
    if task_scores is not None:
        group_comp_results[group_key][task_name].append(task_scores)

print(f"   Player 1 scores stored in '{group_key}' group")

# Analyze Player 2
print(f"\nAnalyzing Player 2...")
player_num = 2
comp_results_p2 = step7_comparison.run_comparative_analysis(
    player2_results['epochs_binned'], 
    player2_results['full_df'], 
    player_num
)

# Determine group
group_key = 'Winner' if player2_results['is_winner'] else 'Loser'
for task_name, task_scores in comp_results_p2.items():
    if task_scores is not None:
        group_comp_results[group_key][task_name].append(task_scores)

print(f"   Player 2 scores stored in '{group_key}' group")

In [ ]:
# =================================================================
# 8. PLOT COMPARATIVE RESULTS
# =================================================================
print(f"\n{'='*60}")
print("PLOTTING COMPARATIVE RESULTS")
print('='*60)

# Create time axis (20 bins = 0-5 seconds)
time_axis = np.linspace(0, 5.0, 20)

# Plot for Winners
print("\nPlotting Winner group...")
step7_comparison.plot_grand_average_comparison(
    group_comp_results['Winner'], 
    "Winner"
)

# Plot for Losers
print("\nPlotting Loser group...")
step7_comparison.plot_grand_average_comparison(
    group_comp_results['Loser'], 
    "Loser"
)

print("\n✅ All analyses complete!")

In [ ]:
# =================================================================
# 9. QUICK SUMMARY
# =================================================================
print(f"\n{'='*60}")
print("SUMMARY STATISTICS")
print('='*60)

# Winner/Loser counts
n_winners = len(group_comp_results['Winner']['Own Current'])
n_losers = len(group_comp_results['Loser']['Own Current'])
print(f"Winners: {n_winners}, Losers: {n_losers}")

# Mean accuracies
if 'player1_scores' in locals() and 'player2_scores' in locals():
    print(f"\nPlayer 1 Mean Accuracy: {np.mean(player1_scores):.2f}%")
    print(f"Player 2 Mean Accuracy: {np.mean(player2_scores):.2f}%")
    
    # Above chance?
    chance_level = 33.33
    p1_above_chance = np.mean(player1_scores) > chance_level
    p2_above_chance = np.mean(player2_scores) > chance_level
    print(f"\nAbove Chance Level ({chance_level}%)?")
    print(f"Player 1: {'YES' if p1_above_chance else 'NO'}")
    print(f"Player 2: {'YES' if p2_above_chance else 'NO'}")

print(f"\n✅ Pipeline testing complete for {SUBJECT_ID}!")